# BIM RAG — Full Ingestion Pipeline

The single notebook to run to make a new IFC file fully usable by the app. One cell at the
bottom, one argument. Everything below is idempotent: re-running it on the same file repairs or
validates missing artifacts without duplicating database data.

Each run performs and then verifies:

1. **Structured import** — `ifc_to_db()` parses the IFC and upserts entities, relationships, and
   relationship members into PostgreSQL, scoped by `source_model_id` (derived from the file's
   SHA-256 fingerprint).
2. **Semantic manifest** — the same `ifc_to_db()` call generates
   `model_semantics/{source_model_id}/{fingerprint}.semantic.json`: a deterministic, LLM-free
   inventory of every queryable concept in the model, plus the concepts it cannot reliably answer.
   This is what the query pipeline's binder reads to understand how *this* model is represented.
3. **RAG documents + vectors** — pgvector setup and `BAAI/bge-m3` embeddings into `rag_documents`.
4. **3D viewer artifact** — IFC to That Open Fragments `.frag` via the frontend's
   `npm run prepare:model`, written to `model_assets/{source_model_id}/{fingerprint}.frag`.
5. **Readiness checks** — database rows, a fingerprint-matched and hash-valid semantic manifest,
   vectors, and the viewer artifact are each confirmed before the model is called query-ready.

The manifest is generated by the production pipeline. This notebook only *displays and verifies*
it — there is deliberately no second implementation here.

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

from sqlalchemy import create_engine, text

import bim_rag
from bim_rag.config import get_db_url, get_model_semantics_root
from bim_rag.pipeline_structured import ifc_to_db
from bim_rag.reporting import print_report
from bim_rag.semantic_manifest import manifest_path, read_manifest

# ingestion/src/bim_rag/__init__.py -> parents[2] == ingestion/
INGESTION_ROOT = Path(bim_rag.__file__).resolve().parents[2]
REPO_ROOT = INGESTION_ROOT.parent
IFC_ORIGINAL_DIR = INGESTION_ROOT / "ifc_original"
FRONTEND_DIR = REPO_ROOT / "frontend"
MODEL_SEMANTICS_ROOT = get_model_semantics_root()
MODEL_ASSETS_DIR = REPO_ROOT / "model_assets"

## Readiness verification

Confirms the four artifacts a model needs before the app can answer questions about it. Reports
every problem it finds rather than stopping at the first, so one run tells you everything that is
missing.

In [ ]:
def verify_model_readiness(source_model_id: int) -> dict:
    """Check database rows, semantic manifest, vectors, and viewer artifact.

    Returns a dict with a per-artifact verdict and a combined `ready` flag.
    The semantic check is the strict one: the artifact must exist at the path
    derived from the model's CURRENT fingerprint and must pass structural
    validation, so a stale manifest left over from an earlier version of the
    same file is reported as missing rather than silently trusted.
    """
    checks: dict[str, object] = {}
    problems: list[str] = []

    engine = create_engine(get_db_url())
    with engine.connect() as conn:
        row = conn.execute(
            text(
                "SELECT file_name, file_fingerprint FROM ifc_source_models WHERE id = :id"
            ),
            {"id": source_model_id},
        ).fetchone()
        if row is None:
            return {"ready": False, "problems": [f"source model {source_model_id} not found"]}
        file_name, fingerprint = row

        entities = conn.execute(
            text("SELECT count(*) FROM ifc_entities WHERE source_model_id = :id"),
            {"id": source_model_id},
        ).scalar()
        relationships = conn.execute(
            text("SELECT count(*) FROM ifc_relationships WHERE source_model_id = :id"),
            {"id": source_model_id},
        ).scalar()
        rag_docs = conn.execute(
            text("SELECT count(*) FROM rag_documents WHERE source_model_id = :id"),
            {"id": source_model_id},
        ).scalar()

    checks["file_name"] = file_name
    checks["entities"] = entities
    checks["relationships"] = relationships
    checks["rag_documents"] = rag_docs
    if not entities:
        problems.append("no entities imported")
    if not rag_docs:
        problems.append("no rag_documents / vectors generated")

    # --- semantic manifest -------------------------------------------------
    path = manifest_path(MODEL_SEMANTICS_ROOT, source_model_id, fingerprint)
    if not path.is_file():
        checks["semantic_manifest"] = "MISSING"
        problems.append(f"no semantic manifest for fingerprint {fingerprint[:16]}...")
    else:
        try:
            document = read_manifest(path)
            identity = document["identity"]
            checks["semantic_manifest"] = {
                "path": str(path.relative_to(REPO_ROOT)),
                "fingerprint_matches": identity["file_fingerprint"] == fingerprint,
                "content_hash": identity["content_hash"][:16] + "...",
                "schema_version": identity["manifest_schema_version"],
                "builder_version": identity["builder_version"],
                "classes": len(document["content"]["object_level"]["classes"]),
                "missing_capabilities": len(
                    document["content"]["global_level"]["missing_capabilities"]
                ),
            }
            if identity["file_fingerprint"] != fingerprint:
                problems.append("semantic manifest fingerprint does not match the source model")
        except Exception as exc:
            checks["semantic_manifest"] = f"INVALID: {exc}"
            problems.append(f"semantic manifest failed validation: {exc}")

    # --- viewer artifact ---------------------------------------------------
    frag = MODEL_ASSETS_DIR / str(source_model_id) / f"{fingerprint}.frag"
    checks["viewer_artifact"] = str(frag.relative_to(REPO_ROOT)) if frag.is_file() else "MISSING"
    if not frag.is_file():
        problems.append("no 3D viewer artifact for this fingerprint")

    checks["ready"] = not problems
    checks["problems"] = problems
    return checks

## Pipeline function

One argument: the IFC file address. Accepts an absolute path, or just a filename that lives in
`ingestion/ifc_original/` (e.g. `"FOJAB_Landsarkivet.ifc"`).

In [ ]:
def run_full_ingestion(ifc_file_address: str) -> dict:
    """Run the full ingestion pipeline for one IFC file, then verify readiness.

    Steps 1-2 are performed by the production `ifc_to_db()` call — structured
    import, semantic manifest, and vectors. Step 3 builds the viewer artifact.
    Step 4 verifies all four before declaring the model query-ready.

    Args:
        ifc_file_address: Absolute path to an .ifc file, or a filename located
            in ingestion/ifc_original/.

    Returns:
        dict with the ingestion report, the resolved source_model_id, and the
        readiness verdict.
    """
    ifc_path = Path(ifc_file_address)
    if not ifc_path.is_file():
        candidate = IFC_ORIGINAL_DIR / ifc_path.name
        if candidate.is_file():
            ifc_path = candidate
        else:
            raise FileNotFoundError(f"IFC source not found: {ifc_file_address}")

    print(f"=== Step 1/3: Structured import + semantic manifest + embeddings ({ifc_path.name}) ===")
    t0 = time.time()
    db_report = ifc_to_db(str(ifc_path))
    print_report(db_report, label="Structured Import + Semantic + Embedding Report")
    print(f"Step 1/3 elapsed: {time.time() - t0:.1f}s")

    source_model_id = db_report["source_model_id"]

    print(f"\n=== Step 2/3: Semantic manifest summary (model_id={source_model_id}) ===")
    if db_report.get("manifest_validated"):
        print(f"  path             : {db_report['manifest_path']}")
        print(f"  source_model_id  : {source_model_id}")
        print(f"  fingerprint      : {db_report['file_fingerprint_prefix']}")
        print(f"  content hash     : {db_report['manifest_content_hash']}")
        print(f"  schema/builder   : {db_report['manifest_schema_version']}"
              f" / {db_report['manifest_builder_version']}")
        print(f"  semantic records : {db_report['manifest_semantic_record_count']}")
        print(f"  classes          : {db_report['manifest_class_count']}")
        print(f"  property fields  : {db_report['manifest_property_field_count']}")
        print(f"  estimated tokens : {db_report['manifest_estimated_tokens']:,}")
        unsupported = db_report["manifest_unsupported_structure_count"]
        print(f"  unsupported      : {unsupported} container(s)")
        print(f"  missing caps     : {db_report['manifest_missing_capability_count']}")
        print("  validation       : PASSED")
    else:
        print(f"  validation       : FAILED - {db_report.get('manifest_error')}")

    print(f"\n=== Step 3/3: 3D viewer artifact (model_id={source_model_id}) ===")
    t1 = time.time()
    cmd = f'npm run prepare:model -- --input "{ifc_path}" --model-id {source_model_id}'
    result = subprocess.run(
        cmd,
        cwd=str(FRONTEND_DIR),
        shell=True,
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(
            f"3D artifact preparation failed (exit code {result.returncode}) "
            f"for model_id={source_model_id}"
        )
    print(f"Step 3/3 elapsed: {time.time() - t1:.1f}s")

    print(f"\n=== Readiness check (model_id={source_model_id}) ===")
    readiness = verify_model_readiness(source_model_id)
    for key, value in readiness.items():
        if key in ("ready", "problems"):
            continue
        print(f"  {key:18}: {value}")

    if readiness["ready"]:
        print("\n=== Model is fully query-ready ===")
    else:
        print("\n=== Model is NOT query-ready ===")
        for problem in readiness["problems"]:
            print(f"  - {problem}")
        print("Re-running this cell is safe and will repair the missing artifacts.")

    return {
        "db_report": db_report,
        "source_model_id": source_model_id,
        "readiness": readiness,
    }

## Run it

To ingest a new IFC file, drop it in `ingestion/ifc_original/` and change the argument below to its
filename (or full path). After this cell finishes and reports **fully query-ready**, the model is
queryable and viewable in the app.

In [ ]:
result = run_full_ingestion("FOJAB_Landsarkivet.ifc")